# Imports and advanced data storage

To add even more functionality to smart contracts, you can import exported functions from other contracts. Xian also supports dynamic dispatch through `importlib`, but this notebook focuses on direct contract imports and richer hash values.


In [ ]:
import_this = """
@export
def dumb_func():
    return 4
"""

to_import = """
import con_import_this

@export
def dumber_func():
    return con_import_this.dumb_func()
"""


In [ ]:
from contracting.client import ContractingClient

client = ContractingClient(signer="stu")
client.flush()
client.submit(import_this, name="con_import_this")
client.submit(to_import, name="con_to_import")


In [ ]:
ti_contract = client.get_contract("con_to_import")
ti_contract.dumber_func()


Direct imports work at the contract level. You still cannot use `from contract import function` syntax, but you can import whole contracts or use `importlib.call(...)` when you need controlled dynamic dispatch.


## Advanced Data
Imagine a game where you can buy or sell pixels. Each pixel has an X and a Y coordinate. At each pixel, we want to store the owner, the price, and the color. While this might sound complicated, it is actually extremely straight forward to do.

All `Hash` objects have the ability to store up to 16 dimensions of information in a key that a max of 1024 bytes in size. The stored object is just JSON using the standard Python JSON encoder and decoder, so you can store things such as lists, sets, etc. At this point in time, there is no limit on how large the value for a key can be, but it will be capped in the future. The current floating figure is 256 bytes to 1024 bytes, so design your contracts accordingly.

Unlike normal Python dictionaries, which `Hash` objects are similar to, you can add different dimensions to your `Hash` object without an issue. For example:

```
h = Hash()
h['one'] = 15
h['one', 'two'] = 20
h['two'] = 25
h['a', 'b', 'c', 'd', 'e', 'f', 'g'] = 6
```

Let's try to build that pixel application.

In [ ]:
coin_contract_source = """
balances = Hash(default_value=0)
token_name = "Stubucks"
token_symbol = "SBX"

@construct
def seed():
    balances[ctx.caller] = 1_000_000


@export
def transfer(amount: int, to: str):
    assert balances[ctx.caller] >= amount, "You don't have enough to spend!"
    balances[ctx.caller] -= amount
    balances[to] += amount


@export
def allow(amount: int, spender: str):
    balances[ctx.caller, spender] = amount


@export
def spend_on_behalf(amount: int, owner: str, to: str):
    assert balances[owner, ctx.caller] >= amount, "You can't spend that!"
    balances[owner, ctx.caller] -= amount
    balances[owner] -= amount
    balances[to] += amount
"""

pixel_game_contract_source = """
import con_coin

pixels = Hash(default_value=None)
max_x = 256
max_y = 256
color_min = 0
color_max = 16

@export
def buy_pixel(x: int, y: int, color: int, amount: int):
    assert 0 <= x < max_x, "X out of bounds!"
    assert 0 <= y < max_y, "Y out of bounds!"
    assert color_min <= color < color_max, "Color is out of bounds!"

    pixel = pixels[x, y]
    if pixel is not None:
        assert amount > pixel["amount"], (
            f"You must pay at least {pixel['amount']} to purchase."
        )

    overwrite_pixel(x, y, color, amount, ctx.caller)


def overwrite_pixel(x, y, color, amount, owner):
    con_coin.spend_on_behalf(amount=amount, owner=ctx.caller, to=ctx.this)
    pixels[x, y] = {
        "owner": owner,
        "amount": amount,
        "color": color,
    }
"""


To keep things simple, let's just assume that we can buy pixels. If you see `overwrite_pixel`, you'll notice that we are setting the X and Y coordinates to a Python dictionary rather than just a primitive type. When we access the database again to pull it out, it is decoded into a Python dictionary again, so you can access it like how you are used to.

Let's try to buy a pixel!

In [ ]:
client.submit(coin_contract_source, name="con_coin")
client.submit(pixel_game_contract_source, name="con_pixel_game")


In [ ]:
coin_contract = client.get_contract("con_coin")
pixel_contract = client.get_contract("con_pixel_game")


In [ ]:
coin_contract.balances['stu'] # Let's make sure we have the coins to make a pixel purchase.

In [ ]:
coin_contract.allow(amount=1, spender="con_pixel_game")
pixel_contract.buy_pixel(x=10, y=10, color=5, amount=1)


In [ ]:
coin_contract.balances['stu'] # The balance has deducted properly

In [ ]:
pixel_contract.pixels[10, 10] # Now we can access the information and recieve the dictionary back

In [ ]:
type(pixel_contract.pixels[10, 10]) # Proof the dictionary is in fact a dictionary!